In [ ]:
# ===============================================================
# SECTION 0 — ENVIRONMENT SETUP
# ===============================================================

!pip install -q "transformers>=4.43.0" "datasets>=2.20.0" peft accelerate bitsandbytes einops

from google.colab import drive
drive.mount('/content/drive')

import os
import re
import math
import pandas as pd
from typing import List, Tuple

import torch
from datasets import load_dataset, concatenate_datasets, Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model


# ===============================================================
# SECTION 1 — REPO CLONE & PATH CONFIG
# ===============================================================

!git clone https://github.com/Jovi3/Time_Series_Project.git
%cd Time_Series_Project
!git checkout update-tokens

PROJECT_DIR = "/content/Time_Series_Project"
JSONL_DIR = PROJECT_DIR    # json files live at root
MASTER_CSV = os.path.join(PROJECT_DIR, "market_features_master.csv")

RESULTS_DIR = "/content/drive/MyDrive/phi3_time_series/results"
CHECKPOINT_DIR = "/content/drive/MyDrive/phi3_time_series/checkpoints"

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("📁 Ready in:", PROJECT_DIR)


# ===============================================================
# SECTION 2 — SERIES + DATE CONFIG
# ===============================================================

SERIES_LIST = [
    "SP500_ret", "NASDAQ_ret", "SPY_ret", "QQQ_ret",
    "VTI_ret", "IVV_ret", "ARKK_ret"
]

ENCODINGS = ["gruver", "delphyne"]

TRAIN_START = "2015-01-02"
TRAIN_END   = "2020-01-01"
TEST_START  = "2021-01-04"
TEST_END    = "2022-01-01"

WINDOW = 60    # sliding window length


# ===============================================================
# SECTION 3 — DATE-BASED SAMPLE COUNTING
# ===============================================================

def get_train_test_counts_from_master():
    df = pd.read_csv(MASTER_CSV)
    df["Date"] = pd.to_datetime(df["Date"])

    train_df = df[(df["Date"] >= TRAIN_START) & (df["Date"] <= TRAIN_END)]
    test_df  = df[(df["Date"] >= TEST_START)  & (df["Date"] <= TEST_END)]

    train_samples = max(0, len(train_df) - WINDOW)
    test_samples  = max(0, len(test_df) - WINDOW)

    print("TRAIN rows:", len(train_df))
    print("TEST rows:", len(test_df))
    print("TRAIN samples:", train_samples)
    print("TEST samples:", test_samples)

    return train_samples, test_samples


def load_encoding_dataset(encoding: str):
    datasets = []
    for series in SERIES_LIST:
        path = os.path.join(JSONL_DIR, f"dataset_{encoding}_{series}.jsonl")
        print("Loading:", path)

        ds = load_dataset("json", data_files=path, split="train")

        def add_meta(ex, idx):
            return {**ex, "series_name": series, "sample_idx": idx}

        ds = ds.map(add_meta, with_indices=True)
        datasets.append(ds)

    return concatenate_datasets(datasets)


def date_based_split_per_series(full_ds: Dataset):
    TRAIN_SAMPLES, TEST_SAMPLES = get_train_test_counts_from_master()

    train_ds = full_ds.filter(lambda x: x["sample_idx"] < TRAIN_SAMPLES)
    test_ds  = full_ds.filter(lambda x: x["sample_idx"] >= TRAIN_SAMPLES)

    print(f"Final split → Train={len(train_ds)}, Test={len(test_ds)}")
    return train_ds, test_ds


# ===============================================================
# SECTION 4 — PROMPT + TOKENIZATION
# ===============================================================

MAX_LEN = 256

def build_prompt(ex, encoding):
    return (
        "You are an assistant that predicts financial time-series returns.\n"
        f"Asset: {ex['series_name']}\n"
        f"Encoding: {encoding}\n"
        "Task: Predict the next return given the sequence.\n\n"
        f"Sequence: {ex['input_text']}\n"
        "Next:"
    )

def format_dataset_for_training(ds, encoding):
    def _fmt(ex):
        prompt = build_prompt(ex, encoding)
        ex["prompt"] = prompt
        ex["text"]   = prompt + " " + ex["target_text"]
        return ex
    return ds.map(_fmt)

def tokenize_dataset(ds, tokenizer):
    def _tok(batch):
        texts = [str(t) for t in batch["text"]]   # ensure strings
        out = tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN
        )
        out["labels"] = out["input_ids"].copy()
        return out
    return ds.map(_tok, batched=True)


# ===============================================================
# SECTION 5 — PHI-3 LoRA INITIALIZATION
# ===============================================================

def init_phi3_with_lora(output_subdir):
    model_name = "microsoft/Phi-3-mini-4k-instruct"

    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

    print("Loading Phi-3-mini…")
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb,
        device_map="auto"
    )

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token

    lora_cfg = LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules=[
            "q_proj","k_proj","v_proj","o_proj",
            "gate_proj","up_proj","down_proj"
        ],
        task_type="CAUSAL_LM"
    )

    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()

    out_dir = os.path.join(CHECKPOINT_DIR, output_subdir)
    os.makedirs(out_dir, exist_ok=True)
    return model, tokenizer, out_dir


# ===============================================================
# SECTION 6 — TRAINING LOOP (WITH FIX)
# ===============================================================

def train_for_encoding(encoding):
    print("\n==============================================")
    print(f"TRAINING ENCODING: {encoding.upper()}")
    print("==============================================")

    full_ds = load_encoding_dataset(encoding)
    train_raw, test_raw = date_based_split_per_series(full_ds)

    train_fmt = format_dataset_for_training(train_raw, encoding)
    test_fmt  = format_dataset_for_training(test_raw, encoding)

    model, tokenizer, out_dir = init_phi3_with_lora(f"phi3_{encoding}_lora")

    train_tok = tokenize_dataset(train_fmt, tokenizer)
    test_tok  = tokenize_dataset(test_fmt, tokenizer)

    # ------------- FIX: REMOVE NON-TENSOR COLUMNS --------------
    cols_to_remove = [
        "input_text", "target_text", "prompt", "text",
        "series_name", "sample_idx"
    ]

    train_tok = train_tok.remove_columns(
        [c for c in cols_to_remove if c in train_tok.column_names]
    )
    test_tok = test_tok.remove_columns(
        [c for c in cols_to_remove if c in test_tok.column_names]
    )
    # ------------------------------------------------------------

    collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

    args = TrainingArguments(
        output_dir=out_dir,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=8,
        num_train_epochs=2,
        learning_rate=2e-4,
        logging_steps=20,
        save_steps=400,
        save_total_limit=2,
        remove_unused_columns=False,
        bf16=torch.cuda.is_available(),
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        tokenizer=tokenizer,
        args=args,
        train_dataset=train_tok,
        eval_dataset=test_tok,
        data_collator=collator
    )

    trainer.train()

    trainer.save_model(out_dir)
    tokenizer.save_pretrained(out_dir)
    print("Saved model to:", out_dir)

    return model, tokenizer, test_fmt, out_dir


# ===============================================================
# SECTION 7 — NUMERIC DECODING
# ===============================================================

def extract_numeric_from_delphyne(text):
    m = re.search(r"[-+]?\d*\.\d+|[-+]?\d+", text)
    return float(m.group()) if m else None

def extract_numeric_from_gruver(text, decimals=6):
    tokens = text.strip().split()
    if not tokens:
        return None
    sign = -1 if tokens[0] == "-" else 1
    digits = [t for t in tokens if t.isdigit()]
    if not digits:
        return None
    int_part = digits[0]
    frac = "".join(digits[1:])[:decimals]
    return sign * float(f"{int_part}.{frac}") if frac else sign * float(int_part)

def decode_prediction_to_float(text, encoding):
    return (
        extract_numeric_from_delphyne(text)
        if encoding == "delphyne"
        else extract_numeric_from_gruver(text)
    )


# ===============================================================
# SECTION 8 — FORECASTING
# ===============================================================

@torch.no_grad()
def generate_next_return(model, tokenizer, prompt, encoding):
    inp = tokenizer(
        str(prompt),
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LEN
    ).to(model.device)

    out = model.generate(
        **inp,
        max_new_tokens=16,
        temperature=0.0,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    gen = out[0, inp["input_ids"].shape[1]:]
    txt = tokenizer.decode(gen, skip_special_tokens=True)
    return decode_prediction_to_float(txt, encoding)


def compute_mse_mae(y_t, y_p):
    d = [(a-b) for a,b in zip(y_t,y_p)]
    mse = sum(x*x for x in d) / len(d)
    mae = sum(abs(x) for x in d) / len(d)
    return mse, mae


def evaluate_encoding_on_test(model, tokenizer, test_ds, encoding):
    rows = []

    for series in SERIES_LIST:
        print("\nEvaluating:", series)

        ds = test_ds.filter(lambda x: x["series_name"] == series)
        ds = ds.sort("sample_idx")
        n = len(ds)
        if n == 0:
            continue

        # -------- 1-STEP --------
        y_true, y_pred = [], []

        for ex in ds:
            y = decode_prediction_to_float(ex["target_text"], encoding)
            p = generate_next_return(model, tokenizer, ex["prompt"], encoding)
            if y is not None and p is not None:
                y_true.append(y)
                y_pred.append(p)

        if y_true:
            mse, mae = compute_mse_mae(y_true, y_pred)
            rows.append({
                "encoding": encoding,
                "series_name": series,
                "horizon": 1,
                "mse": mse, "mae": mae
            })
            print(f"1-step: MSE={mse:.6f}, MAE={mae:.6f}")

        # -------- 3-STEP approx --------
        if n >= 3:
            mse_list, mae_list = [], []
            for i in range(n-2):
                ex0, ex1, ex2 = ds[i], ds[i+1], ds[i+2]
                gt = [
                    decode_prediction_to_float(ex0["target_text"], encoding),
                    decode_prediction_to_float(ex1["target_text"], encoding),
                    decode_prediction_to_float(ex2["target_text"], encoding),
                ]
                if None in gt:
                    continue

                seq = ex0["input_text"]
                preds = []
                for step in range(3):
                    p = generate_next_return(
                        model, tokenizer,
                        build_prompt({"series_name": series, "input_text": seq}, encoding),
                        encoding
                    )
                    preds.append(p)
                    seq = seq + " | " + ex0["target_text"]

                mse3, mae3 = compute_mse_mae(gt, preds)
                mse_list.append(mse3)
                mae_list.append(mae3)

            if mse_list:
                rows.append({
                    "encoding": encoding,
                    "series_name": series,
                    "horizon": 3,
                    "mse": sum(mse_list)/len(mse_list),
                    "mae": sum(mae_list)/len(mae_list)
                })

    return pd.DataFrame(rows)


# ===============================================================
# SECTION 9 — RUN TRAINING + EVAL
# ===============================================================

all_results = []

for encoding in ENCODINGS:
    model, tokenizer, test_fmt, out_dir = train_for_encoding(encoding)
    df_eval = evaluate_encoding_on_test(model, tokenizer, test_fmt, encoding)
    all_results.append(df_eval)

results_df = pd.concat(all_results, ignore_index=True)

csv_path = os.path.join(RESULTS_DIR, "phi3_results.csv")
results_df.to_csv(csv_path, index=False)
print("\n📊 Saved:", csv_path)
print(results_df)


# ===============================================================
# SECTION 10 — PLOTTING
# ===============================================================

import matplotlib.pyplot as plt

for horizon in [1,3]:
    sub = results_df[results_df["horizon"] == horizon]
    if sub.empty:
        continue

    for metric in ["mse", "mae"]:
        plt.figure(figsize=(10,5))
        pivot = sub.pivot(index="series_name", columns="encoding", values=metric)
        pivot.plot(kind="bar", ax=plt.gca())
        plt.title(f"{metric.upper()} — Horizon {horizon}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        out = os.path.join(RESULTS_DIR, f"{metric}_h{horizon}.png")
        plt.savefig(out)
        plt.show()
        print("📈 Saved:", out)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 42.2 MB/s eta 0:00:00
Mounted at /content/drive
Cloning into 'Time_Series_Project'...
remote: Enumerating objects: 73, done.
remote: Counting objects: 100% (73/73), done.
remote: Compressing objects: 100% (55/55), done.
remote: Total 73 (delta 22), reused 66 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (73/73), 5.28 MiB | 11.67 MiB/s, done.
Resolving deltas: 100% (22/22), done.
/content/Time_Series_Project
Branch 'update-tokens' set up to track remote branch 'update-tokens' from 'origin'.
Switched to a new branch 'update-tokens'
📁 Ready in: /content/Time_Series_Project

TRAINING ENCODING: GRUVER
Loading: /content/Time_Series_Project/dataset_gruver_SP500_ret.jsonl


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1702 [00:00<?, ? examples/s]

Loading: /content/Time_Series_Project/dataset_gruver_NASDAQ_ret.jsonl


Generating train split: 0 examples [00:00, ? examples/s]

...

trainable params: 8,912,896 || all params: 3,829,992,448 || trainable%: 0.2327


Map:   0%|          | 0/8379 [00:00<?, ? examples/s]

Map:   0%|          | 0/3535 [00:00<?, ? examples/s]

/tmp/ipython-input-1043683406.py:247: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
20,1.019900
